In [1]:
from printrun.printcore import printcore
import time
from subprocess import PIPE, Popen
import subprocess
import sys

### preconditions
usb port connect von windows aus
wsl usb port accept mit 2 befehlen 


GCode
G90: absolute koordinaten, G91: relative Koordinaten ab aktueller Position
M84: Motoren aus
G0 Y1: Gehe zu position y0, andere achsen nicht ändern
G28 Y: Home Y Achse (vgl. G0 Y0)

In [2]:
txt_location = "C:\\Users\\mah19\\OneDrive\\Desktop\\tmp.txt"
csv_location = "C:\\Users\\mah19\\OneDrive\\Desktop\\tmp.csv"
pm_startup_duration = 3

In [3]:
def send_command(process, command):
    process.stdin.write(command)
    process.stdin.flush()
    process.stdin.write(b"\n")
    process.stdin.flush()

In [4]:
def init_proxmark():
    pm = subprocess.Popen(["wsl"], stdin=PIPE, stdout=PIPE)
    send_command(pm, b'cd ~/source/GITHUB/RfidResearchGroup/proxmark3 && pwd')
    send_command(pm, b'./pm3')
    time.sleep(pm_startup_duration)
    return pm

def run_reader(command):
    pm = init_proxmark()
    send_command(pm, command)
    print(command)
    
    stdout, _ = pm.communicate()
    stdout_str = stdout.decode('utf-8')
    lines = stdout_str.splitlines()
    for line in lines:
        if "successes" in line:
            print(line[4:7])
            return(line[4:7])
        
    return "000"

In [5]:
def init_printer():
    printer = printcore('COM7',250000)
    while not printer.online:
        time.sleep(0.1)
    return printer
  
def home(printer):
    printer.send_now("M84") #home bed
    input("Move bed to 0 and then press enter...")

def move(printer, i):
    print("moving " + str(i) + "mm")
    printer.send_now('G91')
    printer.send_now('G1 Y' + str(i))
    time.sleep(i / 10)

def cleanup_printer(printer):  
    printer.disconnect()

In [6]:
33/10

3.3

# Printer Test

In [29]:
# home(printer)
printer.send_now('M84')
# printer.send_now('G91')
# printer.send_now('G90')
# printer.send_now('G1 Y10')

# HF

In [32]:
printer = init_printer()

In [33]:
home(printer)
start_distance_org = int(input("Ab wann interessanter Bereich org:"))
start_distance_adpt = int(input("Ab wann interessanter Bereich adpt:"))
offset = int(input("Wie dick ist das material:"))

print("Starting measurements...\r\n")
print("org from " + str(start_distance_org))
print("adpt from " + str(start_distance_adpt))
print("material is " + str(offset))

headers_org = []
headers_adpt = []
adpt = []
org = []
printerpos = 0
i = start_distance_org
orgOutOfRange = False
adptOutOfRange = False
doOrg = True
doAdpt = True

try:
    while(True):
        if(len(org) >= 3 and org[-1] == '000' and org[-2] == '000' and org[-3] == '000'):
            orgOutOfRange = True
            print("nothing to do for org")
        if(len(adpt) >= 3 and adpt[-1] == '000'and adpt[-2] == '000' and adpt[-3] == '000'):
            adptOutOfRange = True
            print("nothing to do for adpt")
        if(orgOutOfRange and adptOutOfRange):
            break
        
        movement = abs(printerpos - i)
        printerpos += movement
        move(printer, movement)
        print("printer now at " + str(printerpos))
                
        if not adptOutOfRange and doAdpt and i >= start_distance_adpt:
            successes = run_reader(b'hf 14a readerbaadpt')
            adpt.append(successes)
            headers_adpt.append(str(i + offset).zfill(3))
        if not orgOutOfRange and doOrg and i >= start_distance_org:
            successes = run_reader(b'hf 14a readerbaorg')
            org.append(successes)
            headers_org.append(str(i + offset).zfill(3))

        i = i + 1
            
except KeyboardInterrupt:
    print('interrupted!')

print("adpt")
print(",".join([str(dst).zfill(3) for dst in headers_adpt]))
print(",".join(adpt))
print("org")
print(",".join([str(dst).zfill(3) for dst in headers_org]))
print(",".join(org))

Starting measurements...

org from 35
adpt from 35
material is 10
moving 35mm
printer now at 35
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 36
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 37
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 38
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 39
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 40
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 41
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 42
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 43
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
100
moving 1mm
printer now at 44
b'hf 14a readerbaadpt'
100
b'hf 14a readerbaorg'
098
moving 1mm
printer now at 45
b'hf 14a readerbaadpt'
095
b'hf 14a readerbaorg'
078
moving 1mm
printer now at 46
b'

In [ ]:
print("Done, going to cleanup...")
cleanup_printer(printer)

# LF

In [ ]:
printer = init_printer()

In [ ]:
home(printer)
start_distance_org = int(input("Ab wann interessanter Bereich Org:"))
start_distance_adpt = int(input("Ab wann interessanter Bereich Adpt:"))

print("org from " + str(start_distance_org))
print("adpt from " + str(start_distance_adpt))
print("Starting measurements...\r\n")

headers = []
adpt = []
org = []
printerpos = 0
i = start_distance_org
orgOutOfRange = False
adptOutOfRange = False
orgDetailedMeasurements = False
adptDetailedMeasurements = False
doOrg = True
doAdpt = True
try:
    while(True):
        if(len(org) >= 3 and org[-1] == '000' and org[-2] == '000' and org[-3] == '000'):
            orgOutOfRange = True
            print("nothing to do for org")
        if(len(adpt) >= 3 and adpt[-1] == '000'and adpt[-2] == '000' and adpt[-3] == '000'):
            adptOutOfRange = True
            print("nothing to do for adpt")
        if(orgOutOfRange and adptOutOfRange):
            break
        
        movement = abs(printerpos - i)
        printerpos += movement
        move(printer, movement)
        print("printer now at " + str(printerpos))
        
        if not adptOutOfRange and doAdpt and i >= start_distance_adpt:
            successes = "000"
            if(not adptDetailedMeasurements):
                successes = run_reader(b'lf searchbaadpt')
                if(successes != '010'):
                    adptDetailedMeasurements = True
            if(adptDetailedMeasurements):
                successes = run_reader(b'lf searchbaadpt -l')
            adpt.append(successes)
        if not orgOutOfRange and doOrg and i >= start_distance_org:
            successes = "000"
            if(not orgDetailedMeasurements):
                successes = run_reader(b'lf searchbaorg')
                if(successes != '010'):
                    orgDetailedMeasurements = True
            if(orgDetailedMeasurements):
                successes = run_reader(b'lf searchbaorg -l')
            org.append(successes)
        headers.append(str(i).zfill(3))
        i = i + 1
        
except KeyboardInterrupt:
    print('interrupted!')

print("adpt")
print(",".join([str(dst).zfill(3) for dst in headers]))
print(",".join(adpt))
print("org")
print(",".join([str(dst).zfill(3) for dst in headers]))
print(",".join(org))

In [ ]:
print("test")

In [ ]:
print("Done, going to cleanup...")
cleanup_printer(printer)